# Model Comparison

Compares every regression model on the **same data** and ranks them, so you can see
which performs best. Fair evaluation via **leak-free cross-validation**
(`fs.leakfree_cv`): the training data is resampled *inside* each fold, and each model
is scored on un-resampled validation rows -- the same honest metric used in
`GBR.ipynb` / `ModelSelection.ipynb`.

All models use sensible **default hyperparameters** here (fast, apples-to-apples).
For the tuned result of a single model, use `ModelSelection.ipynb`.

**Read the ranking from `CV_R2`** (mean 5-fold R2). `Test_R2` is also shown but is on
the extreme min/max hold-out split, which unfairly punishes non-tree models that
extrapolate -- so use it only as a secondary signal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import functions as fs
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

SEED = 42

file_path = 'docs/Data_base.xlsx'
df = pd.read_excel(file_path, index_col=0)

element_names = [c for c in df.columns if c not in ('invT', 'kp')]
composition_features = [e for e in element_names if df[e].nunique() > 1] + ['invT']
print('Features (%d):' % len(composition_features), composition_features)

In [ ]:
trainset, testset = fs.data_split(df, element_names, 0.2)

comp_major_low, comp_major_high, comp_major_inter = -0.1, 100.3, 10
comp_minor_low, comp_minor_high, comp_minor_inter = -0.1, 50.3, 0.1
T_low, T_high, T_inter = 10, 2510, 50
size = 8
sampling_params = dict(comp_major_low=comp_major_low, comp_major_high=comp_major_high, comp_major_inter=comp_major_inter,
                       comp_minor_low=comp_minor_low, comp_minor_high=comp_minor_high, comp_minor_inter=comp_minor_inter,
                       T_low=T_low, T_high=T_high, T_inter=T_inter, size=size)

# resampled train set + hold-out test set, for the (secondary) test-set score
Sampled_trainset = fs.data_sampling(trainset, comp_major_low, comp_major_high, comp_major_inter,
                                    comp_minor_low, comp_minor_high, comp_minor_inter,
                                    T_low, T_high, T_inter, size, element_names, random_state=SEED)
sel_tr = Sampled_trainset.loc[:, Sampled_trainset.columns.intersection(composition_features)]
sel_te = testset.loc[:, testset.columns.intersection(composition_features)]
X_train = sel_tr[composition_features]; y_train = np.log10(Sampled_trainset['kp'])
X_test  = sel_te[composition_features]; y_test  = np.log10(testset['kp'])
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

In [ ]:
# Each entry is a zero-arg factory returning a fresh, unfitted model with default params.
# Libraries are imported lazily, so an uninstalled backend is simply skipped below.
def make_factories():
    from sklearn.ensemble import (GradientBoostingRegressor, HistGradientBoostingRegressor,
                                  RandomForestRegressor, ExtraTreesRegressor)
    from sklearn.svm import SVR
    from sklearn.kernel_ridge import KernelRidge
    from sklearn.neural_network import MLPRegressor
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import RBF, WhiteKernel
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    fac = {
        'GBR':          lambda: GradientBoostingRegressor(n_estimators=400, max_depth=3,
                                                          learning_rate=0.05, random_state=SEED),
        'HistGB':       lambda: HistGradientBoostingRegressor(random_state=SEED),
        'RandomForest': lambda: RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
        'ExtraTrees':   lambda: ExtraTreesRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
        # scale-sensitive models -> StandardScaler in front
        'SVR':  lambda: make_pipeline(StandardScaler(), SVR(C=10, gamma='scale')),
        'KRR':  lambda: make_pipeline(StandardScaler(), KernelRidge(alpha=1.0, kernel='rbf')),
        'MLPR': lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(100,),
                                       max_iter=2000, early_stopping=True, random_state=SEED)),
        'GPR':  lambda: make_pipeline(StandardScaler(), GaussianProcessRegressor(
                                       kernel=RBF() + WhiteKernel(), normalize_y=True, random_state=SEED)),
    }
    # optional gradient-boosting backends
    try:
        from xgboost import XGBRegressor
        fac['XGBoost'] = lambda: XGBRegressor(n_estimators=400, random_state=SEED, n_jobs=-1,
                                              objective='reg:squarederror', verbosity=0)
    except Exception:
        print('(XGBoost not installed - skipped)')
    try:
        from lightgbm import LGBMRegressor
        fac['LightGBM'] = lambda: LGBMRegressor(n_estimators=400, random_state=SEED, n_jobs=-1, verbose=-1)
    except Exception:
        print('(LightGBM not installed - skipped)')
    try:
        from catboost import CatBoostRegressor
        fac['CatBoost'] = lambda: CatBoostRegressor(iterations=400, random_state=SEED, verbose=0)
    except Exception:
        print('(CatBoost not installed - skipped)')
    return fac

factories = make_factories()
print('Models to compare:', list(factories))

In [ ]:
rows = []
for name, make in factories.items():
    try:
        cv = fs.leakfree_cv(make, trainset, element_names, composition_features,
                            sampling_params, n_splits=5, seed=SEED)
        m = make(); m.fit(X_train, y_train); pred = m.predict(X_test)
        rows.append({'model': name,
                     'CV_R2': cv['test_R2'].mean(), 'CV_R2_std': cv['test_R2'].std(),
                     'CV_MAE': cv['test_MAE'].mean(), 'CV_MSE': cv['test_MSE'].mean(),
                     'Test_R2': r2_score(y_test, pred), 'Test_MAE': mean_absolute_error(y_test, pred)})
        print('%-13s CV R2=%.3f' % (name, cv['test_R2'].mean()))
    except Exception as e:
        print('%-13s FAILED: %s' % (name, e))

results = pd.DataFrame(rows).sort_values('CV_R2', ascending=False).reset_index(drop=True)
print()
print(results.round(3).to_string(index=False))
best = results.iloc[0]
print('\nBest model by CV R2: %s (CV R2 = %.3f)' % (best['model'], best['CV_R2']))

In [ ]:
order = results.sort_values('CV_R2')  # ascending so best ends up on top
plt.figure(figsize=(8, max(4, 0.5 * len(order))))
plt.barh(order['model'], order['CV_R2'], xerr=order['CV_R2_std'], color='#4C72B0')
plt.xlabel('Leak-free 5-fold CV R2  (higher = better)')
plt.title('Model comparison (default hyperparameters)')
plt.xlim(min(0, order['CV_R2'].min() - 0.05), 1.0)
for y, (r2) in enumerate(order['CV_R2']):
    plt.text(r2, y, ' %.3f' % r2, va='center')
plt.tight_layout()
plt.show()

# Note: ANN (PyTorch) is trained separately in otherModels/ANN.ipynb (CV R2 ~0.84);
# it is not in this sklearn loop.